<a href="https://colab.research.google.com/github/theotheo46/nlp/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22hw4_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Домашнее задание 4: Рекуррентные нейронные сети

В этом задании вам предстоит самостоятельно реализовать модель GRU для решения задачи классификации с пересекающимися классами (multi-label classification). Это вид классификации, в которой каждый объект может относиться одновременно к нескольким классам. Данная задача может возникнуть при классификации фильмов по жанрам, научных или новостных статей по темам, музыкальных композиций по инструментам и так далее.

В нашем случае мы будем работать с датасетом биотехнических новостей и классифицировать их по темам. Этот датасет уже предобработан: текст приведен к нижнему регистру, удалена пунктуация, все слова разделены проблелом.

In [6]:
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh

--2025-05-01 13:26:41--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 155472915 (148M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 148.27M   169MB/s    in 0.9s    

2025-05-01 13:26:42 (169 MB/s) - ‘Miniconda3-latest-Linux-x86_64.sh’ saved [155472915/155472915]



In [9]:
!bash Miniconda3-latest-Linux-x86_64.sh



Welcome to Miniconda3 py313_25.3.1-1

In order to continue the installation process, please review the license
agreement.
Please, press ENTER to continue
>>> ^C


In [10]:
!pip list | grep -E "numpy|gensim|numba|tensorflow"

gensim                                4.3.3
numba                                 0.60.0
numba-cuda                            0.2.0
numpy                                 1.26.4
tensorflow                            2.18.0
tensorflow-datasets                   4.9.8
tensorflow_decision_forests           1.11.0
tensorflow-hub                        0.16.1
tensorflow-io-gcs-filesystem          0.37.1
tensorflow-metadata                   1.17.1
tensorflow-probability                0.25.0
tensorflow-text                       2.18.1


In [4]:
!pip install -q numpy==1.26.4

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 9.1.1 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
blis 1.0.2 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
spacy 3.8.5 requires thinc<8.4.0,>=8.3.4, but you have thinc 9.1.1 which is incompatible.


In [11]:
!pip install --upgrade thinc

  Using cached numpy-2.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.5 which is incompatible.
spacy 3.8.5 requires thinc<8.4.0,>=8.3.4, but you have thinc 9.1.1 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.5 which is incompatible.


In [12]:
!pip install --upgrade gensim

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.5
    Uninstalling numpy-2.2.5:
      Successfully uninstalled numpy-2.2.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 9.1.1 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
blis 1.0.2 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
spacy 3.8.5 requires thinc<8.4.0,>=8.3.4, but you have thinc 9.1.1 which is incompatible.


In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import wandb
from sklearn.metrics import f1_score
from collections import Counter
import nltk
from nltk.corpus import stopwords

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!ls -al /content/drive/MyDrive/data/*

-rw------- 1 root root  9352196 May  1 13:02 /content/drive/MyDrive/data/biotech_news.tsv
-rw------- 1 root root    43313 Oct 17  2023 /content/drive/MyDrive/data/data_problem_1.csv
-rw------- 1 root root    49542 Oct 17  2023 /content/drive/MyDrive/data/germancredit_merged.csv
-rw------- 1 root root 28480737 Apr 27 10:35 /content/drive/MyDrive/data/rare_texts_train.dat
-rw------- 1 root root  3214802 Oct 17  2023 /content/drive/MyDrive/data/real_estate_data.csv
-rw------- 1 root root    52325 Apr 15 10:52 /content/drive/MyDrive/data/yelp_test.csv
-rw------- 1 root root 20201333 Apr 15 10:52 /content/drive/MyDrive/data/yelp_train.csv


In [3]:
nltk.download('stopwords')
stop_words = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
dataset = pd.read_csv('/content/drive/MyDrive/data/biotech_news.tsv', sep='\t')
dataset.head()

,text,labels
0,drive your plow over the bones of the dead by ...,other
1,in the recently tabled national budget denel h...,other
2,shares take a break its good for you picture g...,other
3,reso is currently hiring for two positions pro...,other
4,charter buyer club what is the charter buyer c...,other


In [5]:
wandb.init()

wandb: Currently logged in as: theotheo46 (theotheo46-trs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Предобработка лейблов

Как вы модете заметить, лейблы записаны в виде строк, разделенных запятыми. Для работы с ними нам нужно преобразовать их в числа. Так как каждый объект может принадлежать нескольким классам, закодируем лейблы в виде векторов из 0 и 1, где 1 означает, что объект принадлежит соответствующему классу, а 0 – не принадлежит. Имея такую кодировку, мы сможем обучить модель, решая задачу бинарной классификации для каждого класса.

In [6]:
all_labels = set()
for labels in dataset['labels']:
    all_labels |= set(labels.split(', '))

all_labels = sorted(all_labels)

all_labels, len(all_labels)

(['alliance & partnership',
  'article publication',
  'clinical trial sponsorship',
  'closing',
  'company description',
  'department establishment',
  'event organization',
  'executive appointment',
  'executive statement',
  'expanding geography',
  'expanding industry',
  'foundation',
  'funding round',
  'hiring',
  'investment in public company',
  'ipo exit',
  'm&a',
  'new initiatives & programs',
  'new initiatives or programs',
  'other',
  'participation in an event',
  'partnerships & alliances',
  'patent publication',
  'product launching & presentation',
  'product updates',
  'regulatory approval',
  'service & product providing',
  'subsidiary establishment',
  'support & philanthropy'],
 29)

In [7]:
name2id = {name: i for i, name in enumerate(all_labels)}
id2name = {i: name for name, i in name2id.items()}

In [8]:
def binarize_labels(labels):
    numeric_label = np.zeros(len(name2id), dtype=int)
    for name in labels.split(', '):
        numeric_label[name2id[name]] = 1

    return numeric_label

In [9]:
numeric_labels = dataset['labels'].apply(binarize_labels)
numeric_labels[-4:]

,labels
3035,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3036,"[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, ..."
3037,"[0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3038,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."


## Предобработка данных

В этом задании мы будем обучать рекуррентные нейронные сети. Как мы знаем, они работают хуже для длинных текстов. Поэтому, удалим из текстов стоп слова, слишком редкие, а также слишком частые слова. Все эти слова не должны влиять на класс текста.

In [10]:
texts = dataset['text'].apply(lambda x: x.split())

Сразу разделим выборку на обучающую и тестовую, чтобы считать статистики только по обучающей.

In [11]:
from sklearn.model_selection import train_test_split

texts_train, texts_test, y_train, y_test = train_test_split(texts, numeric_labels, test_size=0.2, random_state=0)

__Задание 1.__ Напишите функцию `process_datasets`, которая принимает на вход обучающую выборку `texts_train`, тестовую выборку `texts_test`, а также параметры `min_wf` и `max_wf`. Функция считает встречаемость каждого слова по тренировочной выборке и удаляет из обоих выборок все __стоп слова__, а также слова, которые встречаются меньше `min_wf` раз и больше `max_wf`. Функция возвращает обработанные выборки в том же порядке.

In [ ]:
# import nltk
# nltk.download('stopwords')

In [12]:
def process_datasets(texts_train, texts_test, min_wf, max_wf):
    word_counter = Counter()
    for text in texts_train:
        word_counter.update(Counter(text))

    def process_text(text):
        clean_text = []
        for word in text:
            if word not in stop_words and word in word_counter and min_wf <= word_counter[word] <= max_wf:
                clean_text.append(word)

        return clean_text

    texts_train = [process_text(text) for text in texts_train]
    texts_test = [process_text(text) for text in texts_test]

    return texts_train, texts_test

Удалим все слова, которые встречаются меньше 4 раз в выборке, а также встречаемость которых больше, чем 95% от размера датасета.

In [13]:
texts_train, texts_test = process_datasets(
    texts_train, texts_test, min_wf=4, max_wf=int(0.95 * len(texts_train))
)

Создадим словарь, который будем использовать для конвертации слов в индексы.

In [14]:
from gensim.corpora.dictionary import Dictionary

dictionary = Dictionary(texts_train)
dictionary.add_documents([['PAD', 'UNK']])
len(dictionary)

16480

Видим, что после обработки у нас осталось около 16 тысяч уникальных слов. Это отличное значение, поэтому больше ничего не будем делать с текстами.

In [15]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    """
    Эта функция вызывается при формировании батча в DataLoader.
    Она токенизирует текст и добавляет паддинги.
    """
    texts, labels = zip(*batch)
    pad_token_id = dictionary.token2id['PAD']
    unk_token_id = dictionary.token2id['UNK']
    input_ids = [torch.tensor(dictionary.doc2idx(text, unknown_word_index=unk_token_id)) for text in texts]
    return (
        pad_sequence(input_ids, padding_value=pad_token_id, batch_first=True).long(),
        torch.tensor(labels).float()
    )

In [16]:
dictionary.token2id['PAD']

16478

Обернем обе выборки в DataLoader и перейдем к обучению рекуррентных сетей.

In [16]:
train_dataset = list(zip(texts_train, y_train))
test_dataset = list(zip(texts_test, y_test))

In [18]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, collate_fn=collate_fn, shuffle=True, batch_size=64)
test_loader = DataLoader(test_dataset, collate_fn=collate_fn, shuffle=False, batch_size=64)

## Обучение моделей

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

### RNN

Напомним, что RNN – это простейшая рекуррентрая нейронная сеть и ее блок RNN выглядит таким образом

<img src="https://i.ibb.co/S5gfLzR/rnn.png" alt="drawing" width="400"/>

Его скрытое состояние обновляется по формуле
$
h_t = \sigma(W x_{t-1} + U h_{t-1} + b_h).
$   
А предсказание считается с помощью применения линейного слоя к последнему токену
$
o_T = O h_T + b_o.
$

Ниже представлена реализация RNN, взятая из практической части урока. Вы можете ее модифицировать, например, для ускорения, по своему усмотрению.

In [20]:
from torch import nn


class RNN(nn.Module):
    def __init__(self, vocab_size, n_classes, pad_token_id, input_size=256, hidden_size=256):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, input_size)

        self.hidden_size = hidden_size
        self.hidden_fc = nn.Linear(input_size + hidden_size, hidden_size)
        self.out_fc = nn.Linear(hidden_size, n_classes)

        self.pad_token_id = pad_token_id

    def forward(self, input_ids, init_h=None):
        bs, seq_len = input_ids.shape

        x = self.embedding(input_ids)

        if init_h is None:
            # инициализируем скрытое состояние нулями
            h_t = torch.zeros(bs, self.hidden_size, device=x.device)
        else:
            h_t = init_h

        hidden_states = []
        for t in range(seq_len):
            # обновляем скрытое состояние RNN и сохраняем его в массив
            x_t = x[:, t, :]
            cat_state = torch.cat((x_t, h_t), dim=1)
            h_t = torch.tanh(self.hidden_fc(cat_state))

            hidden_states.append(h_t.unsqueeze(1))

        # применяем линейный слой ко всем скрытым состояниям, чтобы получить логиты
        hidden_states = torch.cat(hidden_states, dim=1)

        sequence_lengths = (input_ids == self.pad_token_id).int().argmax(-1) - 1
        sequence_lengths = sequence_lengths % input_ids.shape[-1]

        last_hidden_states = hidden_states[torch.arange(len(input_ids)), sequence_lengths]
        logits = self.out_fc(last_hidden_states)

        return torch.sigmoid(logits)

Перед тем, как приступить к обучению, нам нужно выбрать метрику оценки качества. Так как в задаче классификации с пересекающимися классами классы часто несбалансированы, чаще всего в качестве метрики берется [F1 score](https://en.wikipedia.org/wiki/F-score).

__Задание 2.__ Напишите функцию `compute_f1`, которая принимает истинные метки и предсказанные и считает среднее значение F1 по всем классам.

$$
F1_{total} = \frac{1}{K} \sum_{k=1}^K F1(Y_k, \hat{Y}_k),
$$
где $Y_k$ – истинные значения для класса k, а $\hat{Y}_k$ – предсказания.

In [19]:
def compute_f1(y_true, y_pred):
    assert y_true.ndim == 2
    assert y_true.shape == y_pred.shape

    return f1_score(y_true, y_pred, average='macro', zero_division=1)

Осталось написать циклы обучения и валидации.

__Здадание 3.__ Допишите функцию `train` для обучения модели. Она принимает 5 параметров:
* `model`
* `dataloader`
* `optimizer`
* `device` – устройство используемое для обучения: `cpu/cuda`.
* `logging` – бинарная переменная. Если `logging = True`, то функция логирует процесс обучения. В противном случае логгирование не производится.

Функция обучает модель в течение одной эпохи на полученном датасете. Ошибка модели считается как средняя ошибка бинарной классификации для каждого класса.

In [20]:
def train(model, dataloader, optimizer, device='cpu', logging=False):
    """
    Обучает модель (model) на всем наборе данных (dataloader).
    """
    model.to(device)
    model.train()
    criterion = nn.BCELoss()

    for input_ids, labels in dataloader:
        preds = model(input_ids.to(device))
        loss = criterion(preds, labels.to(device))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        f1 = compute_f1((preds > 0.5).int().cpu(), labels)

        # логируем значения ошибки и f1
        if logging:
            wandb.log({
                "train_loss": loss.item(),
                "train_f1": f1
            })

__Здадание 4.__ Допишите функцию `evaluate` для тестирования модели. Она принимает 3 параметра: `model`, `dataloader` и `device`. Функция совершает предсказания для всех объектов в `dataloader`, считает по ним значение F1 и возвращает его. Не забывайте, что если считать F1 отдельно для каждого батча, а затем усреднять, то результат будет неверным.

Если вы хотите изменить поведение функции и, например, возвращать значение лосса или логировать F1 внутри, то вы можете это сделать после прохождения проверки.

In [21]:
@torch.inference_mode()
def evaluate(model, dataloader, device='cpu'):
    """
    Тестирует модель (model) на всем наборе данных (dataloader).
    """

    # не забываем переводить в eval режим
    model.to(device)
    model.eval()

    all_predictions = []
    all_labels = []
    for input_ids, labels in dataloader:
        preds = model(input_ids.to(device))

        all_predictions.extend((preds > 0.5).int().cpu())
        all_labels.extend(labels.cpu())

    # усредняем в самом конце, чтобы не зависеть от размера батча
    f1 = compute_f1(torch.stack(all_predictions), torch.stack(all_labels))
    wandb.log({
        "test_f1": f1
    })
    return f1

Как всегда, начать решение задачи стоит с обучения базовой модели, результат которой мы будем улучшать.

__Задание 5.__ Обучите написанную выше модель RNN с помощью реализованных вами функций. Не обязательно обучать модель до полной сходимости, достаточно будет получить F1 больше 0.33. Так как модель очень простая, мы советуем выбирать скорость обучения побольше.

In [33]:
model = RNN(vocab_size=len(dictionary), n_classes=29 , pad_token_id=16478).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

wandb.init(project="nlp_hw_4", name="rnn")

# учим в течение 30 эпох
for epoch in range(40):
    train(model, train_loader, optimizer, device=device, logging=True)
    test_f1 = evaluate(model, test_loader, device=device)
    wandb.log({
        "epoch": epoch
    })
wandb.finish()

torch.save(model.state_dict(), 'rnn.pt')


test_f1,▂▁▂▁▁▁▂▂▃▃▄▃▄▅▅▆▇▇▇████▇▇▇▇▇▇▇██████████
train_f1,▁▁▃▂▃▃▄▃▄▅▅▅▄▅▅▇▇▇▇▇█▇▇▇█▇▇███████▇█████
train_loss,█▄▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_f1,0.32764
train_f1,0.97873
train_loss,0.00869


### GRU

Теперь перейдем к более интересным рекурренным моделям. В практической части урока мы разбирали LSTM и увидели, что она значительно обходит RNN. Вам предлагается предлагается написать и обучить другую популярную рекуррентную модель – GRU. Она объединяет в себе легковестность RNN и идею модуля памяти LSTM и выглядит следующим образом.

<img src="https://i.ibb.co/3FN81P9/gru.png" alt="drawing" width="400"/>

Параметры блока GPU обновляются вот так:
\begin{align}
&z_t =\sigma(W_z x_{t-1} + U_z h_{t-1} + b_z)\\
&r_t =\sigma(W_r x_{t-1} + U_r h_{t-1} + b_r)\\
&\tilde{h}_t = \tanh(W_h x_{t-1} + U_h(r_t \odot h_{t-1}) + b_h)\\
&h_t = (1-z_t) \odot h_{t-1}+ z_t \odot \tilde{h}_t
\end{align}

__Задание 6.__ Реализуйте стандартную GRU, обучите ее с такими же размерами слоев, что и у RNN, и сравните качество. Если вы все сделали правильно, у вас должно получиться F1 больше 0.36. Использовать `nn.GRU` запрещается. Не забудьте про sigmoid на выходе модели.

In [36]:
wandb.init()

In [25]:
class GRULayer(nn.Module):
    def __init__(self, input_size=256, hidden_size=256):
        super().__init__()

        self.z = nn.Linear(hidden_size + input_size, hidden_size)
        self.r = nn.Linear(hidden_size + input_size, hidden_size)
        self.h = nn.Linear(hidden_size + input_size, hidden_size)

        self.hidden_size = hidden_size

    def forward(self, hidden_states, init_h=None):
        bs, seq_len, input_size = hidden_states.shape

        if init_h is None:
            h_t = torch.zeros(bs, self.hidden_size, device=hidden_states.device)
        else:
            h_t = init_h

        updated_hidden_states = []
        for t in range(seq_len):
            x_t = hidden_states[:, t, :]
            cat_state = torch.cat((x_t, h_t), dim=1)

            z_t = torch.sigmoid(self.z(cat_state))
            r_t = torch.sigmoid(self.r(cat_state))

            h_input_state = torch.cat((x_t, r_t * h_t), dim=1)
            h_t_ = torch.tanh(self.h(h_input_state))

            h_t = (1 - z_t) * h_t + z_t * h_t_

            updated_hidden_states.append(h_t.unsqueeze(1))

        updated_hidden_states = torch.cat(updated_hidden_states, dim=1)

        return updated_hidden_states


class GRU(nn.Module):
    def __init__(
        self,
        vocab_size=16480,
        n_classes=29,
        pad_token_id=16478,
        input_size=256,
        hidden_size=256,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, input_size)

        self.gru_layer = GRULayer(input_size, hidden_size)

        self.out_fc = nn.Linear(hidden_size, n_classes)
        self.pad_token_id = pad_token_id
        self.hidden_size = hidden_size

    def forward(self, input_ids, init_h=None):
        bs, seq_len = input_ids.shape

        x = self.embedding(input_ids)

        x = self.gru_layer(x)

        sequence_lengths = (input_ids == self.pad_token_id).int().argmax(-1) - 1
        sequence_lengths = sequence_lengths % input_ids.shape[-1]

        last_hidden_states = x[torch.arange(len(input_ids)), sequence_lengths]
        logits = self.out_fc(last_hidden_states)

        return torch.sigmoid(logits)


In [26]:
device

device(type='cuda')

In [30]:
model = GRU().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


wandb.init(project="nlp_hw_4", name="gru")

# учим в течение 30 эпох
for epoch in range(30):
    train(model, train_loader, optimizer, device=device, logging=True)
    test_f1 = evaluate(model, test_loader, device=device)
wandb.finish()

torch.save(model.state_dict(), 'gru.pt')

test_f1,▁▁▂▁▂▁▂▂▂▂▂▂▃▃▄▄▅▅▆▆▆▆▇▇▇▇▇▇██
train_f1,▁▃▃▃▃▃▂▃▃▂▃▃▄▃▄▄▅▇▆▅▆▇▇▆▆▆▇▇▇▇▇█▇▇██████
train_loss,█▇▇▆▆▆▆▆▆▆▅▅▅▅▄▄▃▄▃▃▃▂▂▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
test_f1,0.38961
train_f1,0.92414
train_loss,0.00947


In [ ]:
model = GRU(...).to(device)
optimizer = ...

__Задание 7.__ В этом задании у вас есть две опции на выбор: добавить __двунаправленность__ для GRU _или_ добавить __многослойность__. Можно сделать и то, и другое, но дополнительных баллов за это мы не дадим, только бесконечный респект. Обе модификации реализуются довольно просто и дают примерно одинаковый прирост в качестве, поэтому мы не будем вдаваться в подробности. У вас должно получиться качество на тестовой выборке не меньше 0.37.

В грейдере мы будем создавать вашу модель следующим образом:

```
model_path = "это мы тут вашу модель положили где-то.pt"
model = GRU()
model.load_state_dict(torch.load(model_path, map_location=device))
```
Для этого вам нужно задать значения по умолчанию для всех параметров вашей модели. Не забудьте про сигмоиду в конце модели!

In [ ]:
# выберите одно из bidirectional=True и num_layers=2
model = GRU(...).to(device)
optimizer = ...

In [32]:
class GRULayer(nn.Module):
    def __init__(self, input_size=256, hidden_size=256):
        super().__init__()

        self.z = nn.Linear(hidden_size + input_size, hidden_size)
        self.r = nn.Linear(hidden_size + input_size, hidden_size)
        self.h = nn.Linear(hidden_size + input_size, hidden_size)

        self.hidden_size = hidden_size

    def forward(self, hidden_states, init_h=None):
        bs, seq_len, input_size = hidden_states.shape

        if init_h is None:
            h_t = torch.zeros(bs, self.hidden_size, device=hidden_states.device)
        else:
            h_t = init_h

        updated_hidden_states = []
        for t in range(seq_len):
            x_t = hidden_states[:, t, :]
            cat_state = torch.cat((x_t, h_t), dim=1)

            z_t = torch.sigmoid(self.z(cat_state))
            r_t = torch.sigmoid(self.r(cat_state))

            h_input_state = torch.cat((x_t, r_t * h_t), dim=1)
            h_t_ = torch.tanh(self.h(h_input_state))

            h_t = (1 - z_t) * h_t + z_t * h_t_

            updated_hidden_states.append(h_t.unsqueeze(1))

        updated_hidden_states = torch.cat(updated_hidden_states, dim=1)

        return updated_hidden_states


class GRU(nn.Module):
    def __init__(
        self,
        vocab_size=16480,
        n_classes=29,
        pad_token_id=16478,
        input_size=256,
        hidden_size=256,
        num_layers=1,
        bidirectional=True,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, input_size)

        sizes = [input_size] + [hidden_size] * num_layers
        self.gru_layers = nn.ModuleList(
            [GRULayer(sizes[i], sizes[i + 1]) for i in range(num_layers)]
        )
        if bidirectional:
            self.reverse_gru_layers = nn.ModuleList(
                [GRULayer(sizes[i], sizes[i + 1]) for i in range(num_layers)]
            )

        self.out_fc = nn.Linear(hidden_size * (bidirectional + 1), n_classes)
        self.pad_token_id = pad_token_id
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

    def forward(self, input_ids, init_h=None):
        bs, seq_len = input_ids.shape

        x = self.embedding(input_ids)

        if self.bidirectional:
            reverse_x = x.flip(dims=(1,))
            for layer in self.reverse_gru_layers:
                reverse_x = layer(reverse_x)

        for layer in self.gru_layers:
            x = layer(x)

        if self.bidirectional:
            x = torch.cat((x, reverse_x), dim=-1)

        sequence_lengths = (input_ids == self.pad_token_id).int().argmax(-1) - 1
        sequence_lengths = sequence_lengths % input_ids.shape[-1]

        last_hidden_states = x[torch.arange(len(input_ids)), sequence_lengths]
        logits = self.out_fc(last_hidden_states)

        return torch.sigmoid(logits)


In [33]:
model = GRU().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


wandb.init(project="nlp_hw_4", name="gru1")

# учим в течение 30 эпох
for epoch in range(30):
    train(model, train_loader, optimizer, device=device, logging=True)
    test_f1 = evaluate(model, test_loader, device=device)
wandb.finish()

torch.save(model.state_dict(), 'gru1.pt')

test_f1,▁▁▂▂▃▄▄▅▆▆▇███████████████████
train_f1,▁▂▃▃▃▄▃▃▃▄▅▆▆▇▆▆▇▇▇▇█▇████▇█████████████
train_loss,█▇▆▆▆▅▄▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_f1,0.36632
train_f1,0.99943
train_loss,0.00233


## Резюме

Если вы добрались досюда, то вы успешно справились со всеми заданиями. Поздравляем!

Вы должны были заметить, что рекуррентные модели, написанные на python, ужасно долго учатся. Все дело в цикле, который их сильно тормозит. Если в будущем вы будете использовать рекуррентные модели, то мы настоятельно рекомендуем брать реализации из pytorch. Они написаны на C и работают намного быстрее.